In [1]:
import asyncio
import time
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_mcp_adapters.client import MultiServerMCPClient


async def init():
    client = MultiServerMCPClient(
        {

            "email": {
                "transport": "http",  # HTTP-based remote server
                # Ensure you start your weather server on port 8000
                "url": "https://n8n.samair.me/mcp/gmailv2",
                "headers": {
                    "Authorization": "Bearer "
                }
            },

        }
    )

    tools = await client.get_tools()



    model = ChatOllama(
        model="functiongemma",
        temperature=0,
        base_url="http://pi5.local:11434",
        # other params...
    )
    SYSTEM_PROMPT = """
    Your are helpful personal assistant, your job is to help your user.


    """

    agent = create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)
    last_step_time = time.perf_counter()
    current_time = time.perf_counter()
    duration = current_time - last_step_time
    print(f"[{duration:.2f}s]")
    async for  chunk in  agent.astream(
        {"messages": [{"role": "user", "content": "Send a message asking if they are availble on weekend for meeting to sameer@samair.me"}]},

        ):
         for step, data in chunk.items():
            current_time = time.perf_counter()
            duration = current_time - last_step_time
            print(f"[{duration:.2f}s] step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")







In [6]:
await init()




Unknown SSE event: endpoint
Session termination failed: 404


[0.00s]
[8.45s] step: model
content: [{'type': 'tool_call', 'id': '4e961a6e-be86-40fe-a4c1-fe84db74822c', 'name': 'Send_a_message_in_Gmail', 'args': {'Message': 'I am available for meetings this weekend.', 'Subject': 'Meeting for Sameer@Samair.Me', 'To': 'Sameer@samair.me'}}]


Unknown SSE event: endpoint
Session termination failed: 404


[13.28s] step: tools
content: [{'type': 'text', 'text': '[{"id":"19b77fdab93ba21e","threadId":"19b77fdab93ba21e","labelIds":["SENT"]}]', 'id': 'lc_b42810ae-26fd-4544-91e5-bf42c66d7f61'}]
[16.98s] step: model
content: [{'type': 'text', 'text': 'Here is the message sent successfully:\n\n**Message:** I am available for meetings this weekend.\n**To:** Sameer@samair.me\n**Subject:** Meeting for Sameer@Samair.Me\n**Message:** I am available for meetings this weekend.'}]
